# 01. 수신 통화 분석 (STT + 상담/비상담 분류)

NAS에 저장된 통화 녹음 파일 중 **수신(외부→상담원) 통화**만 대상으로, 파일명에서 통화시간을 계산하고
STT(faster-whisper) + Ollama(qwen2.5:3b) 로컬 분류로 상담/비상담/광고/불명확을 판정합니다.
발신(상담원→외부) 통화 분석은 별도 노트북에서 다룹니다.

- **입력**: NAS 폴더(`.env`의 `VOICE_NAS_PATH`)의 `.wav`/`.mp3` 파일. 파일명 형식: `{발신}_{수신}_{시작시각}_{종료시각}.wav`
- **출력**: `../data/inbound/` 아래 메타데이터/분류결과/체크포인트/상담파일목록
- **필요 환경**: `.env`의 `VOICE_NAS_PATH`, 로컬 Ollama 서버(`qwen2.5:3b` pull 필요), `faster-whisper`
- **테스트 모드**: `TEST_N`에 숫자를 넣으면 그 개수만 처리합니다 (`None`이면 전체 처리, 수천 건 기준 수 시간~수십 시간 소요될 수 있음)


In [ ]:
import os
import re
import json
from datetime import datetime

import pandas as pd
import requests
from dotenv import load_dotenv
from faster_whisper import WhisperModel

load_dotenv()

# ============================================================
# 0. 설정
# ============================================================
VOICE_NAS_PATH = os.environ.get("VOICE_NAS_PATH", "").strip()
if not VOICE_NAS_PATH:
    raise RuntimeError(
        ".env에 VOICE_NAS_PATH가 설정되지 않았습니다. "
        r"NAS 상의 통화 녹음 폴더 경로(예: \\NAS서버\mslab\DCS)를 .env에 넣어주세요."
    )

DATA_DIR = "../data/inbound"
os.makedirs(DATA_DIR, exist_ok=True)

METADATA_CSV = f"../data/call_metadata.csv"
CHECKPOINT_FILE = f"{DATA_DIR}/inbound_call_checkpoint.json"
CLASSIFICATION_CSV = f"{DATA_DIR}/inbound_call_classification.csv"
CLASSIFICATION_JSON = f"{DATA_DIR}/inbound_call_classification.json"
CONSULT_LIST_CSV = f"{DATA_DIR}/consult_calls.csv"

MODEL_SIZE = "small"              # tiny / base / small
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:3b"       # 3b: 빠름 / 7b: 정확도 우선
CPU_THREADS = 4
MAX_CHARS = 1800                 # 프롬프트 최대 글자 수
MIN_DURATION = 30                # 초 미만은 STT 스킵, 비상담으로 자동 처리
SAVE_EVERY = 20                  # 몇 건마다 체크포인트 저장할지

# 테스트 모드
# TEST_DATE: "20260601"처럼 특정 날짜(YYYYMMDD, 폴더명과 동일)만 테스트하고 싶으면 설정. None이면 날짜 제한 없음
# TEST_N: 정수를 넣으면 그 개수만 처리, None이면 전체 처리 (야간 배치 권장)
TEST_DATE = None
TEST_N = 5


In [ ]:
# ============================================================
# 1. 파일명 기반 메타데이터 파싱
# ============================================================
def parse_call_metadata(base_path: str) -> pd.DataFrame:
    """NAS 폴더를 순회하며 파일명에서 발신/수신, 통화시간을 추출합니다.
    파일명 형식: {발신번호}_{수신번호}_{시작YYYY_MM_DD_HH_MM_SS}_{종료YYYY_MM_DD_HH_MM_SS}.wav
    """
    records = []
    for root, _dirs, files in os.walk(base_path):
        folder_name = os.path.basename(root)
        if not (len(folder_name) == 8 and folder_name.isdigit()):
            continue
        for fname in files:
            name, ext = os.path.splitext(fname)
            if ext.lower() not in (".wav", ".mp3"):
                continue

            parts = name.split("_")
            caller = parts[0] if len(parts) > 0 else ""
            receiver = parts[1] if len(parts) > 1 else ""

            start_dt = end_dt = duration_sec = None
            if len(parts) >= 14:
                try:
                    start_dt = datetime.strptime("_".join(parts[2:8]), "%Y_%m_%d_%H_%M_%S")
                    end_dt = datetime.strptime("_".join(parts[8:14]), "%Y_%m_%d_%H_%M_%S")
                    duration_sec = (end_dt - start_dt).total_seconds()
                except Exception:
                    pass

            records.append({
                "folder": folder_name,
                "filename": fname,
                "filepath": os.path.join(root, fname),
                "caller": caller,
                "receiver": receiver,
                "start_dt": start_dt,
                "end_dt": end_dt,
                "duration_sec": duration_sec,
                "direction": "발신" if caller == "401" else "수신",
            })
    return pd.DataFrame(records)


df = parse_call_metadata(VOICE_NAS_PATH)
df.to_csv(METADATA_CSV, index=False, encoding="utf-8-sig")

print(f"총 파일: {len(df)}개")
print(df["direction"].value_counts().to_string())
print(f"\n메타데이터 저장 완료 → {METADATA_CSV}")


In [ ]:
# ============================================================
# 2. STT + Ollama 분류 함수
# ============================================================
print(f"Whisper 모델 로딩 중... ({MODEL_SIZE})")
whisper_model = WhisperModel(MODEL_SIZE, device="cpu", compute_type="int8", cpu_threads=CPU_THREADS)
print("Whisper 모델 로딩 완료")

try:
    resp = requests.get("http://localhost:11434/api/tags", timeout=5)
    available_models = [m["name"] for m in resp.json().get("models", [])]
    print(f"Ollama 연결 OK | 사용 가능 모델: {available_models}")
    if OLLAMA_MODEL not in available_models:
        print(f"경고: {OLLAMA_MODEL} 없음 → 'ollama pull {OLLAMA_MODEL}' 실행 필요")
except Exception as e:
    print(f"Ollama 연결 실패: {e}")


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip())


def transcribe(filepath: str) -> str:
    segments, _ = whisper_model.transcribe(
        filepath, language="ko", vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500),
    )
    return normalize_text(" ".join(s.text for s in segments))


def build_prompt(transcript: str) -> str:
    return f"""당신은 기프트코(판촉물 판매 사이트) 상담 전화를 분류하는 전문가입니다.
기프트코는 판촉물(볼펜, 가방, 머그컵, 의류, 인쇄물 등)을 판매하는 B2B/B2C 사이트입니다.

전사문을 읽고 아래 기준으로 분류하세요.

【상담】— 아래 중 하나라도 해당하면 무조건 상담
- 상품 문의 (가격, 수량, 재고, 종류, 샘플 요청)
- 주문/발주/견적/납품/배송/취소/환불 관련
- 결제, 세금계산서, 영수증 관련
- 로그인, 회원가입, 사이트 이용 문의
- 인쇄/제작 파일(디자인, 로고, 일러스트 등) 관련
- 거래처/공급업체 관련 업무 통화
- 문의가 해결되지 않았더라도 업무 관련이면 상담

【비상담】— 아래에만 해당
- 번호를 잘못 눌러 걸린 전화
- 기프트코가 아닌 다른 업체/기관으로 착각한 전화

【광고】— 광고, 영업, 스팸 전화

【불명확】— STT 품질이 너무 낮아 내용 파악 자체가 불가능한 경우만

중요: 업무 내용이 조금이라도 있으면 상담으로 분류하세요.
문제가 해결되지 않았다, 짧은 통화다는 비상담 이유가 될 수 없습니다.

작업:
1. label 결정 (상담/비상담/광고/불명확)
2. 화자를 고객/상담원으로 구분
3. 상담이면 QA 쌍 추출 (챗봇 학습용, 실질적 내용만)
4. 한 줄 요약

규칙: JSON만 반환. 마크다운/코드블록 없이.

JSON:
{{"label":"상담|비상담|광고|불명확","reason":"판단 근거","speaker_dialogue":[{{"speaker":"고객|상담원|불명","text":"내용"}}],"qa_pairs":[{{"question":"질문","answer":"답변"}}],"summary":"요약"}}

전사문:
\"\"\"{transcript}\"\"\"
""".strip()


def safe_parse_json(text: str) -> dict:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    s, e = text.find("{"), text.rfind("}")
    if s != -1 and e > s:
        try:
            return json.loads(text[s:e + 1])
        except Exception:
            pass
    return {}


def normalize_label(raw: str) -> str:
    raw = (raw or "").strip()
    if raw in {"상담", "비상담", "광고", "불명확"}:
        return raw
    if "상담" in raw:
        return "상담"
    if "비상담" in raw or "잘못" in raw:
        return "비상담"
    if "광고" in raw:
        return "광고"
    return "불명확"


def analyze_with_ollama(transcript: str) -> dict:
    prompt = build_prompt(transcript[:MAX_CHARS])
    payload = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": False, "format": "json"}
    resp = requests.post(OLLAMA_URL, json=payload, timeout=(30, 600))
    resp.raise_for_status()
    parsed = safe_parse_json(resp.json().get("response", ""))

    dialogue = [d for d in parsed.get("speaker_dialogue", []) if isinstance(d, dict) and d.get("text")]
    qa_pairs = [q for q in parsed.get("qa_pairs", []) if isinstance(q, dict) and q.get("question") and q.get("answer")]

    return {
        "label": normalize_label(parsed.get("label", "")),
        "reason": parsed.get("reason", ""),
        "summary": parsed.get("summary", ""),
        "speaker_dialogue": dialogue,
        "qa_pairs": qa_pairs,
    }


print("함수 정의 완료")


In [ ]:
# ============================================================
# 3. 수신 통화 분류 배치 실행 (체크포인트 이어받기 + 테스트 모드 지원)
# ============================================================
def classify_inbound_calls():
    inbound_df = df[df["direction"] == "수신"].copy()

    if TEST_DATE:
        inbound_df = inbound_df[inbound_df["folder"] == TEST_DATE]
        print(f"[날짜 필터] {TEST_DATE} 폴더만 대상 → {len(inbound_df)}개")

    short_df = inbound_df[inbound_df["duration_sec"] < MIN_DURATION].copy()
    target_df = inbound_df[inbound_df["duration_sec"] >= MIN_DURATION].copy()

    print(f"전체 파일: {len(df)}개 (수신 {len(df[df['direction']=='수신'])}개 / 발신 {len(df[df['direction']=='발신'])}개)")
    print(f"필터 후 대상: {len(inbound_df)}개 | STT 대상({MIN_DURATION}초 이상): {len(target_df)}개, 스킵(비상담 자동처리): {len(short_df)}개")

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            ckpt = json.load(f)
        done_files = set(ckpt.get("done_files", []))
        all_results = ckpt.get("results", [])
        print(f"체크포인트 복원: {len(done_files)}개 완료")
    else:
        done_files = set()
        all_results = []
        print("새로 시작")

    todo_df = target_df[~target_df["filepath"].isin(done_files)]

    if TEST_N:
        todo_df = todo_df.head(TEST_N)
        short_df = short_df.head(0)  # 테스트 모드에선 단문 통화 추가 안 함
        print(f"[테스트 모드] {TEST_N}건만 처리 (전체 배치는 TEST_N=None으로 변경 후 실행)")

    start_time = datetime.now()
    total = len(todo_df)

    for i, (_, row) in enumerate(todo_df.iterrows(), 1):
        filepath = row["filepath"]
        result_row = {
            "file_name": row["filename"], "filepath": filepath, "folder": row["folder"],
            "direction": row["direction"], "duration_sec": row["duration_sec"],
            "label": "", "reason": "", "summary": "", "transcript": "",
            "speaker_dialogue_text": "", "qa_pairs_text": "", "qa_count": 0,
            "speaker_dialogue": [], "qa_pairs": [],
        }

        try:
            transcript = transcribe(filepath)
        except Exception as e:
            transcript = ""
            print(f"[STT 에러] {row['filename']}: {e}")
        result_row["transcript"] = transcript

        if transcript:
            try:
                analysis = analyze_with_ollama(transcript)
                result_row["label"] = analysis["label"]
                result_row["reason"] = analysis["reason"]
                result_row["summary"] = analysis["summary"]
                result_row["speaker_dialogue_text"] = "\n".join(
                    f"{d['speaker']}: {d['text']}" for d in analysis["speaker_dialogue"]
                )
                result_row["qa_pairs_text"] = "\n\n".join(
                    f"[QA {j}]\nQ: {q['question']}\nA: {q['answer']}"
                    for j, q in enumerate(analysis["qa_pairs"], 1)
                )
                result_row["qa_count"] = len(analysis["qa_pairs"])
                result_row["speaker_dialogue"] = analysis["speaker_dialogue"]
                result_row["qa_pairs"] = analysis["qa_pairs"]
            except Exception as e:
                result_row["label"] = "불명확"
                print(f"[Ollama 에러] {row['filename']}: {e}")
        else:
            result_row["label"] = "불명확 (전사없음)"

        all_results.append(result_row)
        done_files.add(filepath)

        elapsed = (datetime.now() - start_time).total_seconds()
        if i % 5 == 0 or i == total:
            remaining = elapsed / i * (total - i)
            print(f"[{i:4d}/{total}] {result_row['label']:<10} | 경과 {elapsed/60:.1f}분 | "
                  f"남은예상 {remaining/60:.1f}분 | {row['filename'][:40]}")

        if i % SAVE_EVERY == 0 or i == total:
            with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
                json.dump({"done_files": list(done_files), "results": all_results}, f, ensure_ascii=False)
            pd.DataFrame(all_results).to_csv(CLASSIFICATION_CSV, index=False, encoding="utf-8-sig")

    short_rows = [{
        "file_name": r["filename"], "filepath": r["filepath"], "folder": r["folder"],
        "direction": r["direction"], "duration_sec": r["duration_sec"],
        "label": "비상담", "reason": f"{MIN_DURATION}초 미만 통화", "summary": "",
        "transcript": "", "speaker_dialogue_text": "", "qa_pairs_text": "", "qa_count": 0,
        "speaker_dialogue": [], "qa_pairs": [],
    } for _, r in short_df.iterrows()]

    result_df = pd.concat([pd.DataFrame(all_results), pd.DataFrame(short_rows)], ignore_index=True)
    result_df.to_csv(CLASSIFICATION_CSV, index=False, encoding="utf-8-sig")
    result_df.to_json(CLASSIFICATION_JSON, orient="records", force_ascii=False, indent=2)
    print(f"\n완료 → {CLASSIFICATION_CSV}")
    return result_df


result_df = classify_inbound_calls()


In [ ]:
# ============================================================
# 4. 결과 요약 + 상담 파일 목록 추출
# ============================================================
result_df = pd.read_csv(CLASSIFICATION_CSV, encoding="utf-8-sig")

print("=" * 50)
print("[ 분류 결과 요약 ]")
print("=" * 50)
label_counts = result_df["label"].value_counts()
for label, cnt in label_counts.items():
    print(f"  {label:<25}: {cnt:>5}개 ({cnt / len(result_df) * 100:.1f}%)")
print(f"  {'합계':<25}: {len(result_df):>5}개")

result_df["month"] = result_df["folder"].astype(str).str[:6]
print("\n[ 월별 상담 건수 ]")
print(result_df[result_df["label"] == "상담"].groupby("month").size().to_string())

# 상담 파일만 별도 추출 (챗봇 학습 데이터 후보)
consult_df = result_df[result_df["label"] == "상담"][
    ["file_name", "filepath", "direction", "duration_sec", "transcript"]
]
consult_df.to_csv(CONSULT_LIST_CSV, index=False, encoding="utf-8-sig")
print(f"\n상담 파일: {len(consult_df)}개 → {CONSULT_LIST_CSV}")
